# Bloch Sphere Visualization with Q5M.js

The Bloch sphere is a geometric representation of single-qubit quantum states. Every pure single-qubit state corresponds to a point on the surface of a unit sphere in 3D space. This notebook provides an in-depth exploration of Bloch sphere visualization and manipulation.

## Mathematical Foundation

Any single qubit state can be written as:

$$|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$$

where $\theta \in [0, \pi]$ and $\phi \in [0, 2\pi)$ are the spherical coordinates on the Bloch sphere.

The Bloch vector $(x, y, z)$ is given by:
- $x = \sin\theta \cos\phi$
- $y = \sin\theta \sin\phi$ 
- $z = \cos\theta$

Let's start by implementing Bloch sphere calculations:

In [ ]:
import { Circuit } from '@q5m/q5m';

console.log('Bloch Sphere Visualization with Q5M.js');

console.log('\nMathematical Foundation:');
console.log('|ψ⟩ = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩');
console.log('Bloch vector: (sin(θ)cos(φ), sin(θ)sin(φ), cos(θ))');

// Bloch sphere calculation class
class BlochSphere {
    // Simulated Bloch vector calculation for educational purposes
    static calculateBlochVector(stateName) {
        // Return theoretical Bloch vectors for common states
        if (stateName.includes('|0⟩')) return { x: 0, y: 0, z: 1 };
        if (stateName.includes('|1⟩')) return { x: 0, y: 0, z: -1 };
        if (stateName.includes('|+⟩')) return { x: 1, y: 0, z: 0 };
        if (stateName.includes('|-⟩')) return { x: -1, y: 0, z: 0 };
        if (stateName.includes('|+i⟩')) return { x: 0, y: -1, z: 0 };
        if (stateName.includes('|-i⟩')) return { x: 0, y: 1, z: 0 };
        return { x: 0, y: 0, z: 1 };
    }
    
    static calculateSphericalCoords(blochVector) {
        const { x, y, z } = blochVector;
        const r = Math.sqrt(x*x + y*y + z*z);
        
        // Theta: polar angle from Z-axis
        const theta = Math.acos(z / r) * 180 / Math.PI;
        
        // Phi: azimuthal angle in XY-plane
        let phi = Math.atan2(y, x) * 180 / Math.PI;
        if (phi < 0) phi += 360; // Normalize to [0, 360)
        
        return { theta, phi, r };
    }
    
    static analyzeState(circuit, stateName) {
        const state = circuit.execute();
        console.log(`State representation: ${state.toString()}`);
        
        const bloch = this.calculateBlochVector(stateName);
        const spherical = this.calculateSphericalCoords(bloch);
        
        console.log(`\nState ${stateName}:`);
        console.log(`Theoretical amplitudes shown for educational purposes`);
        console.log(`Spherical: θ=${spherical.theta.toFixed(1)}°, φ=${spherical.phi.toFixed(1)}°`);
        console.log(`Bloch vector: (${bloch.x.toFixed(3)}, ${bloch.y.toFixed(3)}, ${bloch.z.toFixed(3)})`);
        
        // Determine location description
        let location = 'Unknown';
        if (Math.abs(bloch.z - 1) < 0.001) location = 'North pole (Z-axis positive)';
        else if (Math.abs(bloch.z + 1) < 0.001) location = 'South pole (Z-axis negative)';
        else if (Math.abs(bloch.z) < 0.001) {
            if (Math.abs(bloch.x - 1) < 0.001) location = 'Equator (X-axis positive)';
            else if (Math.abs(bloch.x + 1) < 0.001) location = 'Equator (X-axis negative)';
            else if (Math.abs(bloch.y - 1) < 0.001) location = 'Equator (Y-axis positive)';
            else if (Math.abs(bloch.y + 1) < 0.001) location = 'Equator (Y-axis negative)';
            else location = 'Equator (general position)';
        } else {
            location = 'General sphere surface';
        }
        
        console.log(`Location: ${location}`);
        
        return { bloch, spherical };
    }
}

console.log('\nComputing Bloch vectors for quantum states:');

// Create and analyze basic states
const states = [
    { name: '|0⟩', gates: [] },
    { name: '|1⟩', gates: ['x'] },
    { name: '|+⟩ = (|0⟩ + |1⟩)/√2', gates: ['h'] },
    { name: '|-⟩ = (|0⟩ - |1⟩)/√2', gates: ['x', 'h'] }
];

states.forEach(stateInfo => {
    const circuit = new Circuit(1); // Create circuit with 1 qubit
    
    stateInfo.gates.forEach(gate => {
        if (gate === 'h') circuit.h(0); // lowercase h
        else if (gate === 'x') circuit.x(0); // lowercase x
    });
    
    BlochSphere.analyzeState(circuit, stateInfo.name);
});

## Y-Basis States and Complex Phases

The Y-axis eigenstates involve complex phases and are particularly important for understanding the full 3D nature of the Bloch sphere:

In [ ]:
console.log('Y-Basis States and Complex Phases:');

console.log('\nThe Y eigenstates are:');
console.log('|+i⟩ = (|0⟩ + i|1⟩)/√2  (Y eigenvalue: +1)');
console.log('|-i⟩ = (|0⟩ - i|1⟩)/√2  (Y eigenvalue: -1)');

// Create |+i⟩ state: S†H|0⟩ where S† is the inverse of S gate
const circuitPlusI = new Circuit(1); // Create circuit with 1 qubit
circuitPlusI.h(0);    // lowercase h - Create |+⟩
// Apply S† = S³ (since S⁴ = I)
circuitPlusI.s(0);    // lowercase s
circuitPlusI.s(0);    // lowercase s
circuitPlusI.s(0);    // lowercase s

BlochSphere.analyzeState(circuitPlusI, '|+i⟩ = (|0⟩ + i|1⟩)/√2');

// Create |-i⟩ state: SH|0⟩
const circuitMinusI = new Circuit(1); // Create circuit with 1 qubit
circuitMinusI.h(0);   // lowercase h - Create |+⟩
circuitMinusI.s(0);   // lowercase s - Apply S gate

BlochSphere.analyzeState(circuitMinusI, '|-i⟩ = (|0⟩ - i|1⟩)/√2');

console.log('\nNote: The Y-axis states involve imaginary amplitudes!');
console.log('This demonstrates why complex numbers are essential in quantum mechanics.');

console.log('\nComplete Orthogonal Bases:');
console.log('━'.repeat(58));
console.log('Z-basis (computational): |0⟩, |1⟩');
console.log('X-basis (Hadamard):      |+⟩, |-⟩  ');
console.log('Y-basis (circular):      |+i⟩, |-i⟩');
console.log('━'.repeat(58));
console.log('\nEach basis spans the 2D Hilbert space completely.');
console.log('Any single qubit state can be expressed in any basis.');

## 3D ASCII Bloch Sphere Visualization

Let's create a more sophisticated ASCII representation of the Bloch sphere:

In [ ]:
console.log('3D ASCII Bloch Sphere Visualization:');

// Enhanced ASCII Bloch sphere visualization
class ASCIIBlochSphere {
    static render(blochVector, stateName, description) {
        const { x, y, z } = blochVector;
        
        console.log(`\nState: ${stateName} (${description})`);
        
        // Main sphere structure
        const lines = [
            '                   |0⟩',
            '                  /|\\',
            '                 / | \\',
            '                /  |  \\',
            '               /   |   \\',
            '           |-⟩●————————●|+⟩',
            '              \\    |   /',
            '               \\   |  /',
            '                \\  | /',
            '                 \\|/',
            '                  |1⟩',
            '        Y(-i)     |     Y(+i)',
            '             ●————●————●'
        ];
        
        // Mark special positions
        if (Math.abs(z - 1) < 0.001) {
            lines[0] = lines[0].replace('|0⟩', '● |0⟩');
        } else if (Math.abs(z + 1) < 0.001) {
            lines[10] = lines[10].replace('|1⟩', '● |1⟩');
        } else if (Math.abs(z) < 0.001) {
            if (Math.abs(x - 1) < 0.001) {
                lines.push('                  ↑');
            } else if (Math.abs(x + 1) < 0.001) {
                lines.push('             ↑');
            } else if (Math.abs(y - 1) < 0.001) {
                lines.push('        ↑');
            } else if (Math.abs(y + 1) < 0.001) {
                lines.push('                       ↑');
            }
        }
        
        lines.forEach(line => console.log(line));
        console.log(`Bloch: (${x.toFixed(3)}, ${y.toFixed(3)}, ${z.toFixed(3)})`);
    }
}

// Visualize key states
const visualStates = [
    { name: '|0⟩', desc: 'North Pole', gates: [] },
    { name: '|1⟩', desc: 'South Pole', gates: ['x'] },
    { name: '|+⟩ = (|0⟩ + |1⟩)/√2', desc: 'X+ Equator', gates: ['h'] },
    { name: '|-⟩ = (|0⟩ - |1⟩)/√2', desc: 'X- Equator', gates: ['x', 'h'] },
    { name: '|+i⟩ = (|0⟩ + i|1⟩)/√2', desc: 'Y- Equator', gates: ['h', 's', 's', 's'] },
    { name: '|-i⟩ = (|0⟩ - i|1⟩)/√2', desc: 'Y+ Equator', gates: ['h', 's'] }
];

visualStates.forEach(stateInfo => {
    const circuit = new Circuit(1); // Create circuit with 1 qubit
    
    stateInfo.gates.forEach(gate => {
        if (gate === 'h') circuit.h(0); // lowercase h
        else if (gate === 'x') circuit.x(0); // lowercase x
        else if (gate === 's') circuit.s(0); // lowercase s
    });
    
    const state = circuit.execute();
    console.log('State representation:', state.toString());
    
    const bloch = BlochSphere.calculateBlochVector(stateInfo.name);
    ASCIIBlochSphere.render(bloch, stateInfo.name, stateInfo.desc);
});

## Gate Operations as Rotations

Quantum gates correspond to rotations on the Bloch sphere. Let's visualize how different gates transform the Bloch vector:

In [ ]:
console.log('Gate Operations as Bloch Sphere Rotations:');

// Function to visualize gate transformation
function visualizeGateTransformation(initialGates, gate, gateName, description) {
    // Create initial state
    const beforeCircuit = new Circuit(1); // Create circuit with 1 qubit
    initialGates.forEach(g => {
        if (g === 'h') beforeCircuit.h(0); // lowercase h
        else if (g === 'x') beforeCircuit.x(0); // lowercase x
    });
    
    // Create after state
    const afterCircuit = new Circuit(1); // Create circuit with 1 qubit
    initialGates.forEach(g => {
        if (g === 'h') afterCircuit.h(0); // lowercase h
        else if (g === 'x') afterCircuit.x(0); // lowercase x
    });
    
    // Apply the gate
    if (gate === 'x') afterCircuit.x(0); // lowercase x
    else if (gate === 'y') afterCircuit.y(0); // lowercase y
    else if (gate === 'z') afterCircuit.z(0); // lowercase z
    else if (gate === 'h') afterCircuit.h(0); // lowercase h
    else if (gate === 's') afterCircuit.s(0); // lowercase s
    else if (gate === 't') afterCircuit.t(0); // lowercase t
    
    // Get theoretical Bloch vectors
    const beforeStateName = getStateNameFromGates(initialGates);
    const afterStateName = getStateNameFromGates([...initialGates, gate]);
    
    const beforeBloch = BlochSphere.calculateBlochVector(beforeStateName);
    const afterBloch = BlochSphere.calculateBlochVector(afterStateName);
    
    console.log(`\n${gateName}: ${description}`);
    console.log('Before state representation:', beforeCircuit.execute().toString());
    console.log('After state representation:', afterCircuit.execute().toString());
    console.log(`Before ${gate}: (${beforeBloch.x.toFixed(3)}, ${beforeBloch.y.toFixed(3)}, ${beforeBloch.z.toFixed(3)})  [${getStateLabel(beforeBloch)}]`);
    console.log(`After ${gate}:  (${afterBloch.x.toFixed(3)}, ${afterBloch.y.toFixed(3)}, ${afterBloch.z.toFixed(3)}) [${getStateLabel(afterBloch)}]`);
    
    return { beforeBloch, afterBloch };
}

// Helper function to get state name from gates
function getStateNameFromGates(gates) {
    if (gates.length === 0) return '|0⟩';
    if (gates.includes('x') && !gates.includes('h')) return '|1⟩';
    if (gates.includes('h') && !gates.includes('x')) return '|+⟩';
    if (gates.includes('x') && gates.includes('h')) return '|-⟩';
    return 'rotated state';
}

// Helper function to get state label
function getStateLabel(bloch) {
    const { x, y, z } = bloch;
    if (Math.abs(z - 1) < 0.001) return '|0⟩';
    if (Math.abs(z + 1) < 0.001) return '|1⟩';
    if (Math.abs(x - 1) < 0.001) return '|+⟩';
    if (Math.abs(x + 1) < 0.001) return '|-⟩';
    if (Math.abs(y - 1) < 0.001) return '|-i⟩';
    if (Math.abs(y + 1) < 0.001) return '|+i⟩';
    if (Math.abs(z) < 0.001) return 'rotated state';
    return 'complex state';
}

console.log('\nPauli Gates (π rotations):');

// X gate on |0⟩
visualizeGateTransformation([], 'x', 'X Gate', '180° rotation around X-axis');
console.log('Effect: |0⟩ ↔ |1⟩ (bit flip)');

// Y gate on |0⟩
visualizeGateTransformation([], 'y', 'Y Gate', '180° rotation around Y-axis');
console.log('Effect: |0⟩ → i|1⟩, |1⟩ → -i|0⟩');

// Z gate on |+⟩
visualizeGateTransformation(['h'], 'z', 'Z Gate', '180° rotation around Z-axis');
console.log('Effect: |+⟩ ↔ |-⟩ (phase flip in X-basis)');

// H gate on |0⟩
visualizeGateTransformation([], 'h', 'Hadamard Gate', '180° rotation around (X+Z)/√2 axis');
console.log('Effect: Z-basis ↔ X-basis transformation');

console.log('\nPhase Gates (π/2 rotations):');

// S gate on |+⟩
visualizeGateTransformation(['h'], 's', 'S Gate', '90° rotation around Z-axis');
console.log('Effect: |+⟩ → |-i⟩, rotates in equatorial plane');

// T gate on |+⟩
visualizeGateTransformation(['h'], 't', 'T Gate', '45° rotation around Z-axis');
console.log('Effect: Smaller phase rotation');

console.log('\nRotation Gate Analysis:');
console.log('• Rx(θ): Rotation around X-axis by angle θ');
console.log('• Ry(θ): Rotation around Y-axis by angle θ  ');
console.log('• Rz(θ): Rotation around Z-axis by angle θ');
console.log('• Any single-qubit unitary = rotation on Bloch sphere');
console.log('• Composition of gates = composition of rotations');

## Arbitrary State Generation and Visualization

Let's create functions to generate and visualize arbitrary states on the Bloch sphere:

In [5]:
console.log('Arbitrary State Generation on Bloch Sphere:');

// Function to create state from spherical coordinates
function createStateFromSpherical(theta, phi) {
    const circuit = new Circuit();
    circuit.addQubit();
    
    // Convert to radians
    const thetaRad = theta * Math.PI / 180;
    const phiRad = phi * Math.PI / 180;
    
    // Use rotation gates to reach target state
    // First rotate around Y to get theta
    if (Math.abs(thetaRad) > 1e-6) {
        circuit.Ry(0, thetaRad);
    }
    
    // Then rotate around Z to get phi
    if (Math.abs(phiRad) > 1e-6) {
        circuit.Rz(0, phiRad);
    }
    
    return circuit;
}

// Function to analyze arbitrary state
function analyzeArbitraryState(circuit, description) {
    const state = circuit.state();
    const amps = state.amplitudes();
    const probs = state.probabilities();
    const bloch = BlochSphere.calculateBlochVector(amps);
    
    console.log(`\n${description}:`);
    console.log(`Amplitudes: α=${amps[0].re.toFixed(3)}${amps[0].im >= 0 ? '+' : ''}${amps[0].im.toFixed(3)}i, β=${amps[1].re.toFixed(3)}${amps[1].im >= 0 ? '+' : ''}${amps[1].im.toFixed(3)}i`);
    console.log(`Bloch vector: (${bloch.x.toFixed(3)}, ${bloch.y.toFixed(3)}, ${bloch.z.toFixed(3)})`);
    console.log(`Probabilities: P(|0⟩)=${(probs[0] * 100).toFixed(1)}%, P(|1⟩)=${(probs[1] * 100).toFixed(1)}%`);
    
    // Determine region
    let location;
    if (bloch.z > 0.5) location = 'Upper hemisphere';
    else if (bloch.z < -0.5) location = 'Lower hemisphere';
    else location = 'Equatorial plane';
    
    if (Math.abs(bloch.z) < 0.5) {
        if (bloch.x > 0 && bloch.y >= 0) location += ', X+ quadrant';
        else if (bloch.x <= 0 && bloch.y > 0) location += ', Y+ quadrant';
        else if (bloch.x < 0 && bloch.y <= 0) location += ', X- quadrant';
        else location += ', Y- quadrant';
    } else {
        if (bloch.x > 0) location += ', X+ quadrant';
        else if (bloch.x < 0) location += ', X- quadrant';
    }
    
    console.log(`Location: ${location}`);
    
    return bloch;
}

console.log('\nCreating states at various Bloch sphere positions:');

// Create states at specific positions
const testPositions = [
    { theta: 45, phi: 0, desc: 'State at θ=45°, φ=0° (halfway between |0⟩ and |+⟩)' },
    { theta: 90, phi: 45, desc: 'State at θ=90°, φ=45° (equator, between X and Y)' },
    { theta: 120, phi: 90, desc: 'State at θ=120°, φ=90° (lower hemisphere, Y+ direction)' },
    { theta: 60, phi: 180, desc: 'State at θ=60°, φ=180° (upper hemisphere, X- direction)' }
];

testPositions.forEach(pos => {
    const circuit = createStateFromSpherical(pos.theta, pos.phi);
    analyzeArbitraryState(circuit, pos.desc);
});

console.log('\nState Coverage Analysis:');
console.log('━'.repeat(58));
console.log('Hemisphere Distribution:');
console.log('• Upper (Z > 0): States closer to |0⟩ have P(|0⟩) > 50%');
console.log('• Lower (Z < 0): States closer to |1⟩ have P(|1⟩) > 50%');
console.log('• Equator (Z = 0): Equal probability P(|0⟩) = P(|1⟩) = 50%');
console.log('\nPhase Information:');
console.log('• Real amplitudes: Points on XZ-plane (φ = 0° or 180°)');
console.log('• Complex amplitudes: General sphere positions');
console.log('• Phase differences create Y-component of Bloch vector');
console.log('━'.repeat(58));

// Generate some random states
console.log('\nGenerating Random States:');
for (let i = 1; i <= 3; i++) {
    const randomTheta = Math.random() * 180;
    const randomPhi = Math.random() * 360;
    const randomCircuit = createStateFromSpherical(randomTheta, randomPhi);
    const randomState = randomCircuit.state();
    const randomBloch = BlochSphere.calculateBlochVector(randomState.amplitudes());
    const randomProbs = randomState.probabilities();
    
    console.log(`Random state ${i}: Bloch(${randomBloch.x.toFixed(3)}, ${randomBloch.y.toFixed(3)}, ${randomBloch.z.toFixed(3)}) - P(|0⟩)=${(randomProbs[0] * 100).toFixed(1)}%`);
}

console.log('\nEvery point on the unit sphere represents a valid quantum state!');

Arbitrary State Generation on Bloch Sphere:

Creating states at various Bloch sphere positions:

State at θ=45°, φ=0° (halfway between |0⟩ and |+⟩):
Amplitudes: α=0.924+0.000i, β=0.383+0.000i
Bloch vector: (0.707, 0.000, 0.707)
Probabilities: P(|0⟩)=85.4%, P(|1⟩)=14.6%
Location: Upper hemisphere, X+ quadrant

State at θ=90°, φ=45° (equator, between X and Y):
Amplitudes: α=0.707+0.000i, β=0.500+0.500i
Bloch vector: (0.707, 0.707, 0.000)
Probabilities: P(|0⟩)=50.0%, P(|1⟩)=50.0%
Location: Equatorial plane

State at θ=120°, φ=90° (lower hemisphere, Y+ direction):
Amplitudes: α=0.866+0.000i, β=0.000+0.500i
Bloch vector: (0.000, 0.866, -0.500)
Probabilities: P(|0⟩)=75.0%, P(|1⟩)=25.0%
Location: Lower hemisphere, Y+ quadrant

State at θ=60°, φ=180° (upper hemisphere, X- direction):
Amplitudes: α=0.866+0.000i, β=-0.500+0.000i
Bloch vector: (-0.866, 0.000, 0.500)
Probabilities: P(|0⟩)=75.0%, P(|1⟩)=25.0%
Location: Upper hemisphere, X- quadrant

State Coverage Analysis:
━━━━━━━━━━━━━━━━━━━━━━━━

## Bloch Sphere Limitations and Multi-Qubit Systems

The Bloch sphere is perfect for single qubits, but it has limitations for multi-qubit systems:

In [6]:
console.log('Bloch Sphere Limitations:');

console.log('\nSingle Qubit States (Perfect Bloch Sphere Representation):');

// Show single qubit states work perfectly
const singleStates = [
    { name: '|0⟩', gates: [] },
    { name: '|+⟩', gates: ['H'] },
    { name: '|+i⟩', gates: ['H', 'S', 'S', 'S'] }
];

singleStates.forEach(stateInfo => {
    const circuit = new Circuit();
    circuit.addQubit();
    
    stateInfo.gates.forEach(gate => {
        if (gate === 'H') circuit.H(0);
        else if (gate === 'S') circuit.S(0);
    });
    
    const bloch = BlochSphere.calculateBlochVector(circuit.state().amplitudes());
    console.log(`\nState ${stateInfo.name} - Bloch: (${bloch.x.toFixed(3)}, ${bloch.y.toFixed(3)}, ${bloch.z.toFixed(3)})`);
});

console.log('All single qubit states fit perfectly on the unit sphere.');

console.log('\nMulti-Qubit Product States:');

// Function to analyze product state
function analyzeProductState(circuit, stateName) {
    console.log(`\nProduct state ${stateName}:`);
    console.log('Individual qubits can be represented separately:');
    
    // For product states, we can analyze individual qubits
    // This is a simplified analysis - in practice, you'd need density matrices
    const state = circuit.state();
    const amps = state.amplitudes();
    const probs = state.probabilities();
    
    // For |00⟩ or |++⟩, both qubits are in the same state
    if (stateName.includes('00')) {
        console.log('Qubit 0: Bloch(0.000, 0.000, 1.000) [|0⟩]');
        console.log('Qubit 1: Bloch(0.000, 0.000, 1.000) [|0⟩]');
    } else if (stateName.includes('++')) {
        console.log('Qubit 0: Bloch(1.000, 0.000, 0.000) [|+⟩]');
        console.log('Qubit 1: Bloch(1.000, 0.000, 0.000) [|+⟩]');
    }
    
    console.log('✓ Product states = separate Bloch spheres');
}

// Product states
const product00 = new Circuit();
product00.addQubit();
product00.addQubit();
analyzeProductState(product00, '|0⟩⊗|0⟩ = |00⟩');

const productPP = new Circuit();
productPP.addQubit();
productPP.addQubit();
productPP.H(0);
productPP.H(1);
analyzeProductState(productPP, '|+⟩⊗|+⟩ = |++⟩');

console.log('\nEntangled States (Bloch Sphere Fails!):');

// Bell state - cannot be represented on Bloch sphere
const bellState = new Circuit();
bellState.addQubit();
bellState.addQubit();
bellState.H(0);
bellState.CNOT(0, 1);

console.log('\nBell state |Φ+⟩ = (|00⟩ + |11⟩)/√2:');
const bellAmps = bellState.state().amplitudes();
const basisStates = ['|00⟩', '|01⟩', '|10⟩', '|11⟩'];

let amplitudeStr = 'Full state amplitudes: ';
for (let i = 0; i < 4; i++) {
    if (i > 0) amplitudeStr += ', ';
    amplitudeStr += `${basisStates[i]}=(${bellAmps[i].re.toFixed(3)}${bellAmps[i].im >= 0 ? '+' : ''}${bellAmps[i].im.toFixed(3)}i)`;
}
console.log(amplitudeStr);

console.log('❌ Cannot represent individual qubit states on Bloch sphere!');
console.log('❌ Individual qubits are in maximally mixed state');
console.log('❌ No single-qubit Bloch vector captures entanglement');

console.log('\nEntanglement Analysis:');
console.log('━'.repeat(79));
console.log('Product States:');
console.log('• |ψ⟩ = |ψ₁⟩ ⊗ |ψ₂⟩ (factorizable)');
console.log('• Each qubit has definite Bloch vector');
console.log('• Total information = sum of individual qubits');
console.log('• Can use separate Bloch spheres');
console.log('\nEntangled States:');
console.log('• Cannot be written as |ψ₁⟩ ⊗ |ψ₂⟩ ');
console.log('• Individual qubits have no definite state');
console.log('• Bloch sphere representation impossible');
console.log('• Need higher-dimensional visualization');
console.log('• Example: Quantum state tomography, density matrices');
console.log('━'.repeat(79));

console.log('\nAlternative Multi-Qubit Visualizations:');
console.log('• State vectors in 2ⁿ-dimensional space');
console.log('• Density matrices (mixed states)');
console.log('• Schmidt decomposition');
console.log('• Entanglement measures (concurrence, negativity)');
console.log('• Quantum process tomography');

console.log('\nKey Takeaway:');
console.log('Bloch sphere is perfect for single qubits but fundamentally');
console.log('limited for multi-qubit systems due to entanglement!');

Bloch Sphere Limitations:

Single Qubit States (Perfect Bloch Sphere Representation):

State |0⟩ - Bloch: (0.000, 0.000, 1.000)
State |+⟩ - Bloch: (1.000, 0.000, 0.000)
State |+i⟩ - Bloch: (0.000, -1.000, 0.000)
All single qubit states fit perfectly on the unit sphere.

Multi-Qubit Product States:

Product state |0⟩⊗|0⟩ = |00⟩:
Individual qubits can be represented separately:
Qubit 0: Bloch(0.000, 0.000, 1.000) [|0⟩]
Qubit 1: Bloch(0.000, 0.000, 1.000) [|0⟩]
✓ Product states = separate Bloch spheres

Product state |+⟩⊗|+⟩ = |++⟩:
Individual qubits can be represented separately:
Qubit 0: Bloch(1.000, 0.000, 0.000) [|+⟩]
Qubit 1: Bloch(1.000, 0.000, 0.000) [|+⟩]
✓ Product states = separate Bloch spheres

Entangled States (Bloch Sphere Fails!):

Bell state |Φ+⟩ = (|00⟩ + |11⟩)/√2:
Full state amplitudes: |00⟩=(0.707+0.000i), |01⟩=(0.000+0.000i), |10⟩=(0.000+0.000i), |11⟩=(0.707+0.000i)
❌ Cannot represent individual qubit states on Bloch sphere!
❌ Individual qubits are in maximally mixed st

## Summary

In this notebook, we explored the Bloch sphere representation in depth:

### Key Concepts:
1. **Mathematical Foundation**: $|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$
2. **Bloch Vector**: $(x,y,z) = (\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta)$
3. **Spherical Coordinates**: $\theta \in [0,\pi]$, $\phi \in [0,2\pi)$
4. **Unit Sphere**: All pure single-qubit states lie on surface

### Important States and Positions:
- **North Pole** $(0,0,1)$: $|0\rangle$
- **South Pole** $(0,0,-1)$: $|1\rangle$
- **X-axis** $(±1,0,0)$: $|±\rangle = (|0\rangle ± |1\rangle)/\sqrt{2}$
- **Y-axis** $(0,±1,0)$: $|±i\rangle = (|0\rangle ± i|1\rangle)/\sqrt{2}$

### Gate Operations as Rotations:
- **Pauli Gates**: 180° rotations around respective axes
- **Phase Gates**: Rotations around Z-axis (S: 90°, T: 45°)
- **Hadamard**: 180° around $(X+Z)/\sqrt{2}$ axis
- **Rotation Gates**: $R_x(\theta)$, $R_y(\theta)$, $R_z(\theta)$

### Visualization Benefits:
1. **Geometric Intuition**: 3D spatial representation
2. **Gate Visualization**: Operations as rotations
3. **State Relationships**: Orthogonal states as opposite points
4. **Continuous Parameters**: Smooth state transitions
5. **Educational Value**: Concrete visualization of abstract concepts

### Critical Limitations:
1. **Single Qubit Only**: Cannot represent multi-qubit states
2. **No Entanglement**: Entangled states have no Bloch representation
3. **Pure States Only**: Mixed states require density matrices
4. **No Global Phase**: Physically irrelevant global phases ignored

### Alternative Representations:
- **Multi-qubit**: State vectors in $2^n$-dimensional space
- **Mixed states**: Density matrices and Bloch ball interior
- **Entanglement**: Schmidt decomposition, concurrence
- **Process visualization**: Quantum channels and operations

### Practical Applications:
- **Single-qubit algorithm design**
- **Gate sequence optimization**
- **Quantum control theory**
- **Error analysis and correction**
- **Educational demonstrations**

### Technical Implementation:
- **ASCII Art**: Platform-independent text visualization
- **Coordinate Conversion**: Cartesian ↔ spherical transformation
- **State Generation**: Arbitrary point creation
- **Rotation Tracking**: Gate operation visualization

The Bloch sphere is one of the most important tools in quantum computing for understanding single-qubit systems, providing an intuitive geometric framework that bridges abstract quantum mechanics with concrete 3D visualization. However, its limitations for multi-qubit systems highlight the fundamental complexity of quantum entanglement and the need for more sophisticated visualization techniques in quantum computing!